# Notebook 02: Preprocessing

**Project:** Lightweight machine-learning-based cybersecurity threat detection for resource-constrained hardware
**Author:** Anush Jindal

Cleans the Wednesday file based on the findings of notebook 01 and splits it into a
training set and a test set:

1. Drop rows with infinite values, rows with missing values, and duplicate rows.
2. Check for identifier columns (IPs, flow IDs, timestamps) that could leak information.
3. Split 80/20 into training and test sets, stratified by class.

No model is trained here. The output is a clean, split dataset that notebooks 03 and 04
use.

## Order of operations and data leakage

Data leakage happens when information from the test set influences training, even
indirectly, which makes the evaluation look better than the model really is. The order
used here is:

```
Raw dataset
   ↓
Row filters that learn nothing from the data
(dropping infinite, missing and duplicate rows)
   ↓
Train/test split
   ↓
Anything fitted to the data (scalers, encoders, imputers, feature selection
by correlation with the label) is fitted on the training set only
   ↓
The same fitted transform is applied to the test set
```

Dropping rows before the split is safe because it is a filter on each row's own values,
not a statistic learned from the dataset. Anything that learns from the data's
distribution must come after the split and be fitted on the training set only.

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Paths: the repo layout when run locally, the Drive folder when run in Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/threat detector")
except ImportError:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_DIR / "data"
RAW_CSV = DATA_DIR / "Wednesday-workingHours.pcap_ISCX.csv"
LABEL_COLUMN = "Label"

df = pd.read_csv(RAW_CSV)
df.columns = df.columns.str.strip()  # same whitespace fix as notebook 01

print("Loaded shape:", df.shape)

## 2. Infinite values

`Flow Bytes/s` and `Flow Packets/s` are `inf` when a flow's duration is 0 (division by
zero). `isnull()` does not detect `inf`, so it is checked separately.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

inf_mask = np.isinf(df[numeric_cols]).any(axis=1)
print(f"Rows containing at least one infinite value: {inf_mask.sum()} out of {len(df)}")
print(f"That's {inf_mask.sum() / len(df) * 100:.3f}% of rows")

1,297 rows (0.19%) contain an infinite value. A rate that is undefined because of a
division by zero has no meaningful replacement value, and the rows are a very small
share of the data, so they are dropped.

In [ ]:
rows_before = len(df)
df = df[~inf_mask].copy()
rows_after = len(df)

print(f"Dropped {rows_before - rows_after} rows containing infinite values.")
print(f"Shape is now: {df.shape}")

## 3. Missing values

In [ ]:
missing = df.isnull().sum()
missing_only = missing[missing > 0].sort_values(ascending=False)
print(f"Columns still containing missing values: {len(missing_only)}")
missing_only

No missing values remain: the 1,008 missing `Flow Bytes/s` values found in notebook 01
were in the rows already dropped for infinite values. The `dropna()` below is kept as a
safeguard and drops 0 rows.

In [ ]:
rows_before = len(df)
df = df.dropna().copy()
rows_after = len(df)

print(f"Dropped {rows_before - rows_after} rows containing missing values.")
print(f"Shape is now: {df.shape}")

## 4. Duplicate rows

In [ ]:
rows_before = len(df)
df = df.drop_duplicates().copy()
rows_after = len(df)

print(f"Dropped {rows_before - rows_after} duplicate rows.")
print(f"Shape is now: {df.shape}")

80,914 duplicate rows are dropped, fewer than the 81,909 counted in notebook 01 because
some duplicates were among the infinite-value rows. 692,703 - 1,297 - 80,914 = 610,492
rows remain.

## 5. Identifier columns

Identifier columns (flow ID, IP addresses, timestamp) describe *which* connection a row
is, not how it behaves. A model trained on them could memorise specific hosts or times
from the training data instead of learning attack behaviour. They are removed if
present.

In [ ]:
IDENTIFIER_COLUMNS_TO_REMOVE = [
    "Flow ID",
    "Source IP",
    "Destination IP",
    "Timestamp",
]

existing_to_remove = [c for c in IDENTIFIER_COLUMNS_TO_REMOVE if c in df.columns]
missing_from_df = [c for c in IDENTIFIER_COLUMNS_TO_REMOVE if c not in df.columns]

print("Will remove:", existing_to_remove)
if missing_from_df:
    print("Not found in this file (skipping):", missing_from_df)

df = df.drop(columns=existing_to_remove)
print("\nShape is now:", df.shape)

This version of the Wednesday file has none of these columns, so nothing is removed and
all 78 features are kept.

`Destination Port` is deliberately not treated as an identifier: unlike an IP address, a
port (22, 80, 443) says which service is targeted and can carry real behavioural signal.
Whether to keep it is a feature-selection decision, made explicitly in notebook 03 and
tested as a possible shortcut in notebook 04. The reasoning for every column is in
`research/feature_documentation.md`.

## 6. Features (X) and label (y)

In [ ]:
X = df.drop(columns=[LABEL_COLUMN])
y = df[LABEL_COLUMN]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 7. Train/test split

The split is made before anything is fitted to the data. `stratify=y` keeps the class
proportions the same in both sets; without it, a random split could leave very few (or
no) rows of a rare attack in the test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

In [ ]:
print("Class balance in y_train:")
print((y_train.value_counts(normalize=True) * 100).round(2))
print("\nClass balance in y_test:")
print((y_test.value_counts(normalize=True) * 100).round(2))

`X_train` is (488393, 78) and `X_test` is (122099, 78). The class percentages match
between the two sets. Heartbleed's 11 rows are split 9 / 2, which is why it cannot be
scored reliably later.

## 8. Fitting transforms later

Decision trees do not need scaling or encoding, so none is applied. Any transform added
later must follow the same rule:

```python
scaler = StandardScaler()
scaler.fit(X_train)                        # learn from the training set only
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)   # same fitted scaler, never re-fitted on test
```

## 9. Save the split

The four files are written to the data folder, where notebook 03 reads them. They are
git-ignored like the raw data.

In [ ]:
X_train.to_csv(DATA_DIR / "X_train.csv", index=False)
X_test.to_csv(DATA_DIR / "X_test.csv", index=False)
y_train.to_csv(DATA_DIR / "y_train.csv", index=False)
y_test.to_csv(DATA_DIR / "y_test.csv", index=False)

print("Saved X_train.csv, X_test.csv, y_train.csv, y_test.csv to", DATA_DIR)

## Summary

| Step | Rows removed | Rows left |
|---|---|---|
| Raw file | | 692,703 |
| Infinite values | 1,297 (0.19%) | 691,406 |
| Missing values | 0 | 691,406 |
| Duplicates | 80,914 | 610,492 |
| Identifier columns | none present | 610,492 x 78 features |
| Split 80/20, stratified | | 488,393 train / 122,099 test |